# Enterprise RAG — Hands-On, Part 7 of 11: Reranking

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.identity import get_principal
from enterprise_rag.authz.policy import compile_prefilter
from enterprise_rag.llm.client import LLMClient
from enterprise_rag.ingest import store

llm = LLMClient()
marco = get_principal("u_marco_t3")
w_marco = compile_prefilter(marco)

# Build the index if it is not already there (same check as Part 4).
from enterprise_rag.ingest.store import collection_stats
try:
    stats = collection_stats("meridian")
    assert stats["chunks"] > 0
except Exception:
    from enterprise_rag.ingest.pipeline import ingest
    print("building index...")
    print(ingest().render())

---
# Part 7 - Reranking

Retrieval optimises for **recall** over a huge corpus, cheaply and approximately. Reranking optimises
for **precision** over ~20 candidates, expensively and accurately. Do both.

Why it works: dense search embeds the question and the document *separately* and never compares them
directly. A reranker sees them **together** and answers one question: *does this passage actually
answer this?*

In [ ]:
from enterprise_rag.retrieval.rerank import LLMReranker

q = "Who has to approve an emergency rate limit override above 2x?"
vec = llm.embed([q])[0]
candidates = store.dense_search(marco.tenant_id, vec, w_marco, 12)

print("BEFORE reranking (vector similarity order):")
for i, sc in enumerate(candidates[:8]):
    print(f"  {i+1}. {sc.chunk.chunk_id:<16}{sc.score:.3f}  {sc.chunk.section[:44]}")

reranked = LLMReranker(llm).rerank(q, candidates, top_k=6)
print("\nAFTER reranking (does this passage answer the question?):")
for i, sc in enumerate(reranked):
    print(f"  {i+1}. {sc.chunk.chunk_id:<16}{sc.rerank_score:>4}/10  {sc.chunk.section[:44]}")

**An interaction worth naming in an interview:** reranking happens *after* ACL enforcement, so a
restricted user's top-6 is the best of *their own* authorised pool - never a diluted version of
someone else's. If you post-filtered instead, you would rerank documents the user cannot see and
then hand them a nearly empty context.

---

**◀ Previous:** [6. Query transformation](part06-query-transformation.ipynb)

**Next ▶:** [8. The full graph](part08-full-graph.ipynb)
